### Desafio Shoply

A Shoply (e-commerce) esta perdendo recompra: 1 em cada 3 clientes que ja comprou nao volta em 90 dias.
Marketing culpa o app, Produto/UX culpa preco e frete. O objetivo aqui e:

1. Diagnostico: entender o que explica a queda de recompra e o aumento do tempo entre pedidos
2. Predicao (opcional): estimar risco de um cliente nao recomprar nos proximos 90 dias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Carregando e limpando a base

In [ ]:
df = pd.read_csv('orders_final.csv')
df.head()

In [ ]:
df.isna().sum()

In [ ]:
#pedido duplicado (mesmo order_id repetido)
df['order_id'].duplicated().sum()

In [ ]:
df = df.drop_duplicates(subset='order_id', keep='first')

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['delivered_at'] = pd.to_datetime(df['delivered_at'], errors='coerce')
df['estimated_delivery_date'] = pd.to_datetime(df['estimated_delivery_date'], errors='coerce')

In [ ]:
def limpar_texto(coluna):
    return coluna.astype(str).str.strip().str.lower().replace('nan', np.nan)

df['order_status'] = limpar_texto(df['order_status'])
df['order_category'] = limpar_texto(df['order_category'])
df['payment_method'] = limpar_texto(df['payment_method'])
df['delivery_state'] = limpar_texto(df['delivery_state']).str.upper()

In [ ]:
#tem varias grafias erradas na forma de pagamento 
pagamento_map = {
    'creditcard': 'credit_card',
    'credit_card': 'credit_card',
    'boleto_bancario': 'boleto',
    'boleto': 'boleto',
    'pixx': 'pix',
    'pix': 'pix',
    'paypal': 'paypal',
}
df['payment_method'] = df['payment_method'].map(pagamento_map)

In [ ]:
estados_validos = ['SP','RJ','MG','ES','PR','SC','RS','DF','GO','MT','MS',
                    'AC','AL','AM','AP','BA','CE','PA','PB','PE','PI','MA','RN','RO','RR','SE','TO']

df.loc[~df['delivery_state'].isin(estados_validos), 'delivery_state'] = np.nan

In [ ]:
df['sku_count'] = df['sku_count'].astype(str).str.strip().replace({'dois': '2'})
df['sku_count'] = df['sku_count'].str.replace(',0', '', regex=False)
df['sku_count'] = pd.to_numeric(df['sku_count'], errors='coerce')
df.loc[df['sku_count'] < 0, 'sku_count'] = np.nan

In [ ]:
def limpar_valor(valor):
    valor = str(valor).strip()
    if ',' in valor:
        partes = valor.split(',')
        if len(partes) > 2:
            valor = partes[0] + '.' + partes[1]
        else:
            valor = valor.replace(',', '.')
    return valor

df['order_value'] = df['order_value'].apply(limpar_valor)
df['order_value'] = pd.to_numeric(df['order_value'], errors='coerce')

In [ ]:
df['discount_value'] = df['discount_value'].fillna(0)

In [ ]:
df.isna().sum()

In [ ]:
regiao_map = {
    'SP': 'sudeste', 'RJ': 'sudeste', 'MG': 'sudeste', 'ES': 'sudeste',
    'PR': 'sul_centrooeste', 'SC': 'sul_centrooeste', 'RS': 'sul_centrooeste',
    'DF': 'sul_centrooeste', 'GO': 'sul_centrooeste', 'MT': 'sul_centrooeste', 'MS': 'sul_centrooeste',
    'AC': 'norte_nordeste', 'AL': 'norte_nordeste', 'AM': 'norte_nordeste', 'AP': 'norte_nordeste',
    'BA': 'norte_nordeste', 'CE': 'norte_nordeste', 'PA': 'norte_nordeste', 'PB': 'norte_nordeste',
    'PE': 'norte_nordeste', 'PI': 'norte_nordeste', 'MA': 'norte_nordeste', 'RN': 'norte_nordeste',
    'RO': 'norte_nordeste', 'RR': 'norte_nordeste', 'SE': 'norte_nordeste', 'TO': 'norte_nordeste',
}
sla_map = {'sudeste': 3, 'sul_centrooeste': 5, 'norte_nordeste': 8}

df['regiao'] = df['delivery_state'].map(regiao_map)
df['sla_padrao'] = df['regiao'].map(sla_map)

In [ ]:
df['atraso_entrega'] = (df['delivered_at'] - df['estimated_delivery_date']).dt.days
df[['regiao', 'sla_padrao', 'atraso_entrega']].describe(include='all')

In [ ]:
data_referencia = pd.Timestamp('2025-12-20')

pedidos_completos = df[df['order_status'] == 'delivered'].copy()

In [ ]:
ultima_compra = pedidos_completos.groupby('customer_id')['order_date'].max()
dias_sem_comprar = (data_referencia - ultima_compra).dt.days

df_clientes = pd.DataFrame({
    'ultima_compra': ultima_compra,
    'dias_sem_comprar': dias_sem_comprar,
})

df_clientes['ativo'] = df_clientes['dias_sem_comprar'] <= 90
df_clientes['churn'] = (df_clientes['dias_sem_comprar'] > 90).astype(int)

df_clientes.head()

In [ ]:
df_clientes['churn'].mean()

In [ ]:
sns.histplot(df_clientes['dias_sem_comprar'], bins=40)
plt.axvline(90, color='r', linestyle='--')
plt.title('Dias desde a ultima compra')

In [ ]:
rfm = pedidos_completos.groupby('customer_id').agg(
    total_pedidos=('order_id', 'count'),
    ticket_medio=('order_value', 'mean'),
    desconto_medio=('discount_value', 'mean'),
    sku_medio=('sku_count', 'mean'),
    atraso_medio=('atraso_entrega', 'mean'),
)

df_clientes = df_clientes.join(rfm)
df_clientes.head()

In [ ]:
colunas_numericas = ['churn', 'total_pedidos', 'ticket_medio', 'desconto_medio', 'sku_medio', 'atraso_medio']

sns.heatmap(df_clientes[colunas_numericas].corr()[['churn']].sort_values(by='churn', ascending=False),
            vmin=-1, vmax=1, annot=True, cmap='BrBG')

In [ ]:
categoria_favorita = pedidos_completos.groupby('customer_id')['order_category'].agg(lambda x: x.value_counts().idxmax())
df_clientes['categoria_favorita'] = categoria_favorita

df_clientes.groupby('categoria_favorita')['churn'].mean().sort_values(ascending=False)

In [ ]:
pagamento_favorito = pedidos_completos.groupby('customer_id')['payment_method'].agg(
    lambda x: x.value_counts().idxmax() if x.notna().any() else np.nan
)
df_clientes['pagamento_favorito'] = pagamento_favorito

df_clientes.groupby('pagamento_favorito')['churn'].mean().sort_values(ascending=False)

In [ ]:
#verificando se entrega atrasada aumenta o churn
df_clientes['atrasado'] = df_clientes['atraso_medio'] > 0
df_clientes.groupby('atrasado')['churn'].mean()

In [ ]:
#verificando se desconto influencia a recompra
faixas_desconto = pd.cut(df_clientes['desconto_medio'], bins=[-1, 0, 10, 50, 1000])
df_clientes.groupby(faixas_desconto)['churn'].mean()

In [ ]:
pedidos_completos = pedidos_completos.sort_values(['customer_id', 'order_date'])
pedidos_completos['dias_desde_pedido_anterior'] = pedidos_completos.groupby('customer_id')['order_date'].diff().dt.days

In [ ]:
tendencia_recompra = pedidos_completos.groupby(pedidos_completos['order_date'].dt.to_period('M'))['dias_desde_pedido_anterior'].mean()

tendencia_recompra.plot(figsize=(10, 4))
plt.title('Tempo medio entre pedidos ao longo do tempo')
plt.ylabel('dias')

In [ ]:
primeira_compra = pedidos_completos.groupby('customer_id')['order_date'].min()

pedidos_completos['mes_pedido'] = pedidos_completos['order_date'].dt.to_period('M')
pedidos_completos['mes_coorte'] = pedidos_completos['customer_id'].map(primeira_compra).dt.to_period('M')
pedidos_completos['indice_coorte'] = (pedidos_completos['mes_pedido'] - pedidos_completos['mes_coorte']).apply(lambda x: x.n)

In [ ]:
coorte = pedidos_completos.groupby(['mes_coorte', 'indice_coorte'])['customer_id'].nunique().reset_index()
tamanho_coorte = coorte[coorte['indice_coorte'] == 0].set_index('mes_coorte')['customer_id']

tabela_retencao = coorte.pivot(index='mes_coorte', columns='indice_coorte', values='customer_id')
tabela_retencao = tabela_retencao.divide(tamanho_coorte, axis=0)

#so os 6 primeiros meses de retencao
tabela_retencao.loc[:, 0:6].head(12)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(tabela_retencao.loc[:, 0:6].head(12), annot=True, fmt='.0%', cmap='BrBG')
plt.title('Retencao por coorte (mes da primeira compra)')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df_modelo = df_clientes.dropna(subset=['ticket_medio', 'desconto_medio', 'sku_medio',
                                        'atraso_medio', 'categoria_favorita', 'pagamento_favorito']).copy()

df_modelo = pd.get_dummies(df_modelo, columns=['categoria_favorita', 'pagamento_favorito'], dtype='int64')

x = df_modelo.drop(columns=['churn', 'ativo', 'ultima_compra', 'dias_sem_comprar'])
y = df_modelo['churn']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.7, random_state=51)

In [ ]:
modelo_churn = LogisticRegression(max_iter=1000).fit(x_train, y_train)
y_pred = modelo_churn.predict(x_test)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
pd.Series(modelo_churn.coef_[0], index=x.columns).sort_values()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
df_segmento = df_clientes.dropna(subset=['total_pedidos', 'ticket_medio', 'desconto_medio',
                                          'sku_medio', 'atraso_medio']).copy()

x = df_segmento[['total_pedidos', 'ticket_medio', 'desconto_medio', 'sku_medio', 'atraso_medio']]
x_padronizado = StandardScaler().fit_transform(x)

In [ ]:
inercias = []
for k in range(1, 8):
    kmeans_teste = KMeans(n_clusters=k, random_state=51, n_init=10).fit(x_padronizado)
    inercias.append(kmeans_teste.inertia_)

plt.plot(range(1, 8), inercias, marker='o')
plt.xlabel('numero de clusters')
plt.ylabel('inercia')
plt.title('Metodo do cotovelo')

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=51, n_init=10).fit(x_padronizado)
df_segmento['cluster'] = kmeans.labels_

df_segmento.groupby('cluster')[['total_pedidos', 'ticket_medio', 'desconto_medio',
                                 'sku_medio', 'atraso_medio', 'churn']].mean()

In [ ]:
df_segmento['cluster'].value_counts()

In [ ]:
sns.scatterplot(data=df_segmento, x='total_pedidos', y='ticket_medio', hue='cluster', palette='deep')
plt.title('Segmentos de cliente (total de pedidos x ticket medio)')